# Investment & Stock Market Analytics with PySpark

**Mục tiêu:** sử dụng PySpark để xử lý dữ liệu thị trường nhiều mã, tính chỉ tiêu lợi nhuận–rủi ro, phân tích tương quan, danh mục và tín hiệu đầu tư cơ bản.

> Dataset đi kèm là **dữ liệu mô phỏng phục vụ giảng dạy**, có schema tương tự dữ liệu OHLCV thực tế. Không dùng kết quả cho quyết định đầu tư thực tế.

## 1. Khởi tạo môi trường

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder.appName("TH05_InvestmentAnalytics").getOrCreate()
spark

## 2. Đọc dữ liệu và kiểm tra chất lượng

In [ ]:
stocks = spark.read.option("header", True).option("inferSchema", True).csv("data/stock_market_daily.csv")
market = spark.read.option("header", True).option("inferSchema", True).csv("data/market_index.csv")
weights = spark.read.option("header", True).option("inferSchema", True).csv("data/sample_portfolio.csv")

stocks.printSchema()
stocks.show(5)
print("Rows:", stocks.count(), "Tickers:", stocks.select("Ticker").distinct().count())

In [ ]:
# Missing values
stocks.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in stocks.columns]).show(truncate=False)

# Duplicates theo Date + Ticker
dup = stocks.groupBy("Date","Ticker").count().filter(F.col("count") > 1)
print("Duplicate keys:", dup.count())

### Bài tập
1. Loại duplicate theo `(Date, Ticker)`.
2. Loại hoặc xử lý các dòng thiếu `Volume`/`Sector`.
3. Kiểm tra điều kiện `High >= max(Open, Close)` và `Low <= min(Open, Close)`.
4. Giải thích vì sao các bước kiểm tra này quan trọng trong dữ liệu tài chính.

In [ ]:
stocks = stocks.dropDuplicates(["Date","Ticker"])     .filter(F.col("Close") > 0)     .filter(F.col("Open") > 0)

stocks = stocks.withColumn("Date", F.to_date("Date"))

## 3. Return Analytics

In [ ]:
w = Window.partitionBy("Ticker").orderBy("Date")
stocks = stocks.withColumn("PrevClose", F.lag("Close").over(w))     .withColumn("DailyReturn", (F.col("Close")/F.col("PrevClose")) - 1)     .withColumn("LogReturn", F.log(F.col("Close")/F.col("PrevClose")))

stocks.select("Date","Ticker","Close","DailyReturn","LogReturn").show(10)

### Bài tập: Lợi nhuận
Tính cho từng mã: số ngày giao dịch, average daily return, cumulative return, annualized return (xấp xỉ), min/max daily return. Xếp hạng 10 mã có cumulative return cao nhất.

In [ ]:
ret_summary = stocks.groupBy("Ticker","Sector").agg(
    F.count("DailyReturn").alias("N"),
    F.avg("DailyReturn").alias("AvgDailyReturn"),
    F.stddev("DailyReturn").alias("DailyVolatility"),
    F.min("DailyReturn").alias("WorstDay"),
    F.max("DailyReturn").alias("BestDay")
).withColumn("AnnualizedReturnApprox", F.col("AvgDailyReturn")*252)  .withColumn("AnnualizedVolatility", F.col("DailyVolatility")*F.sqrt(F.lit(252.0)))
ret_summary.orderBy(F.desc("AnnualizedReturnApprox")).show(10, truncate=False)

## 4. Risk Analytics: Volatility, Sharpe proxy, Drawdown

In [ ]:
risk_free = 0.03
risk = ret_summary.withColumn(
    "SharpeProxy",
    (F.col("AnnualizedReturnApprox")-F.lit(risk_free))/F.col("AnnualizedVolatility")
)
risk.orderBy(F.desc("SharpeProxy")).show(10, truncate=False)

In [ ]:
# Drawdown theo từng mã
cumw = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
stocks = stocks.withColumn("RunningPeak", F.max("Close").over(cumw))     .withColumn("Drawdown", F.col("Close")/F.col("RunningPeak") - 1)
max_dd = stocks.groupBy("Ticker").agg(F.min("Drawdown").alias("MaxDrawdown"))
max_dd.orderBy("MaxDrawdown").show(10)

### Bài tập: Risk Profile
Tạo bảng `Ticker – Annualized Return – Annualized Volatility – SharpeProxy – MaxDrawdown`. Chọn 5 mã hấp dẫn nhất theo quan điểm risk-adjusted return và giải thích.

## 5. Moving Average & Momentum

In [ ]:
w20 = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-19,0)
w60 = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-59,0)
stocks = stocks.withColumn("SMA20", F.avg("Close").over(w20))     .withColumn("SMA60", F.avg("Close").over(w60))     .withColumn("MA_Signal", F.when(F.col("SMA20") > F.col("SMA60"), "Bullish").otherwise("Bearish"))

stocks.select("Date","Ticker","Close","SMA20","SMA60","MA_Signal").show(10)

### Bài tập
1. Xác định số ngày Bullish/Bearish của từng mã.
2. Tạo tín hiệu crossover khi SMA20 cắt lên/cắt xuống SMA60.
3. Không được kết luận “mua/bán chắc chắn”; hãy thảo luận giới hạn của tín hiệu kỹ thuật.

## 6. Sector Analytics

In [ ]:
sector = stocks.groupBy("Date","Sector").agg(F.avg("DailyReturn").alias("SectorReturn"))
sector_summary = sector.groupBy("Sector").agg(
    F.avg("SectorReturn").alias("AvgReturn"),
    F.stddev("SectorReturn").alias("Volatility")
).withColumn("AnnualizedReturn", F.col("AvgReturn")*252)  .withColumn("AnnualizedVolatility", F.col("Volatility")*F.sqrt(F.lit(252.0)))
sector_summary.show(truncate=False)

## 7. Correlation & Diversification

In [ ]:
# Pivot return theo ngày x ticker; chỉ chuyển bảng nhỏ đã tổng hợp sang Pandas để tính correlation/visualization
pivot = stocks.select("Date","Ticker","DailyReturn").groupBy("Date").pivot("Ticker").agg(F.first("DailyReturn")).orderBy("Date")
pdf = pivot.toPandas().set_index("Date")
corr = pdf.corr()
corr.iloc[:8,:8]

### Bài tập
1. Tìm 10 cặp cổ phiếu có correlation cao nhất (không tính chính nó).
2. Tìm 10 cặp correlation thấp nhất.
3. Giải thích diversification dựa trên correlation.
4. Vì sao không nên `toPandas()` với toàn bộ dữ liệu Big Data?

## 8. Portfolio Analytics

In [ ]:
# Danh mục mẫu
weights.show()
portfolio_src = stocks.join(weights, "Ticker", "inner").filter(F.col("DailyReturn").isNotNull())
portfolio_daily = portfolio_src.groupBy("Date").agg(
    F.sum(F.col("Weight")*F.col("DailyReturn")).alias("PortfolioReturn")
).orderBy("Date")
portfolio_daily.show(5)

In [ ]:
port_stats = portfolio_daily.agg(
    F.avg("PortfolioReturn").alias("AvgDailyReturn"),
    F.stddev("PortfolioReturn").alias("DailyVolatility")
).first()
ann_ret = port_stats['AvgDailyReturn']*252
ann_vol = port_stats['DailyVolatility']*(252**0.5)
sharpe = (ann_ret-risk_free)/ann_vol
print("Annualized return:", ann_ret)
print("Annualized volatility:", ann_vol)
print("Sharpe proxy:", sharpe)

### Bài tập: So sánh danh mục
Tạo thêm ít nhất 3 danh mục: Equal Weight, Technology-heavy và Defensive. So sánh return, volatility, Sharpe proxy và Max Drawdown. Đưa ra nhận xét về trade-off giữa lợi nhuận và rủi ro.

## 9. Beta so với thị trường

In [ ]:
market = market.withColumn("Date", F.to_date("Date"))
wm = Window.orderBy("Date")
market = market.withColumn("PrevIndex", F.lag("MarketIndex").over(wm))     .withColumn("MarketReturn", F.col("MarketIndex")/F.col("PrevIndex")-1)
joined = stocks.join(market.select("Date","MarketReturn"), "Date", "inner")

# beta = cov(stock, market) / var(market)
betas = joined.groupBy("Ticker").agg(
    (F.covar_samp("DailyReturn","MarketReturn") / F.var_samp("MarketReturn")).alias("Beta")
)
betas.orderBy(F.desc("Beta")).show(10)

### Bài tập
Giải thích Beta > 1, Beta ≈ 1, Beta < 1. Tìm 5 mã có beta cao nhất/thấp nhất và liên hệ với volatility.

## 10. Spark SQL

In [ ]:
stocks.createOrReplaceTempView("stocks")
spark.sql("""
SELECT Sector,
       AVG(DailyReturn)*252 AS annual_return_approx,
       STDDEV(DailyReturn)*SQRT(252) AS annual_volatility
FROM stocks
WHERE DailyReturn IS NOT NULL
GROUP BY Sector
ORDER BY annual_return_approx DESC
""").show(truncate=False)

### Bài tập: Viết tối thiểu 6 truy vấn Spark SQL
Gợi ý: top return; top volume; sector performance; drawdown lớn; bullish ratio; high-volatility stocks.

## 11. Visualization
Chỉ lấy **dữ liệu đã chọn/tổng hợp** sang Pandas để vẽ.

In [ ]:
import matplotlib.pyplot as plt
sample = stocks.filter(F.col("Ticker").isin("AAPL","MSFT","JPM","JNJ","CAT"))     .select("Date","Ticker","Close").toPandas()
for t,g in sample.groupby("Ticker"):
    plt.plot(g["Date"], g["Close"], label=t)
plt.legend(); plt.title("Selected stock prices"); plt.xlabel("Date"); plt.ylabel("Close"); plt.show()

## 12. Investment Insight & Critical Thinking
Sinh viên viết **Executive Investment Note** gồm:
- 5 phát hiện định lượng;
- 3 nhận xét về risk-return;
- 2 nhận xét về diversification;
- 1 danh mục đề xuất phục vụ mục đích **học thuật/minh họa**;
- ít nhất 3 hạn chế của phân tích.

> **Không** được biến kết quả thực hành thành khuyến nghị đầu tư thực tế.

## Bài tập mở rộng
Thiết kế một **Stock Ranking Score** từ 4 thành phần: annualized return, volatility, Sharpe proxy và Max Drawdown. Chuẩn hóa chỉ tiêu trước khi cộng trọng số. Giải thích vì sao lựa chọn trọng số đó.